# Continue Pre-training GPT-2 on Custom Text

In Session 2 we learned that **pre-training** teaches a model language by predicting the next token on massive text. In this notebook we'll see it firsthand:

1. Load a small pre-trained GPT-2
2. Generate text **before** additional training (baseline)
3. Continue pre-training on a custom text file
4. Generate text **after** training and compare

This is the same process used at scale to build GPT-3, LLaMA, etc. — we're just doing it small enough to run on a laptop.

---

In [ ]:
%pip install -q transformers torch datasets accelerate

In [ ]:
import torch
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    TextDataset,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## 1 — Prepare a Training Corpus

We need a `.txt` file to train on. You can use **any** text:
- A novel from Project Gutenberg
- Your own writing
- Wikipedia articles on a topic

Below we download a small sample from [tiny_shakespeare](https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt) — Shakespeare's collected works (~1 MB). Replace the URL with your own file if you prefer.

In [ ]:
import urllib.request
from pathlib import Path

CORPUS_PATH = Path("corpus.txt")

if not CORPUS_PATH.exists():
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    print(f"Downloading from {url}...")
    urllib.request.urlretrieve(url, CORPUS_PATH)

text = CORPUS_PATH.read_text(encoding="utf-8")
print(f"Corpus: {len(text):,} characters, ~{len(text.split()):,} words")
print(f"\n--- First 500 characters ---")
print(text[:500])

## 2 — Load GPT-2 and Establish a Baseline

Before any additional training, let's see what GPT-2 generates for Shakespeare-style prompts.

In [ ]:
MODEL_NAME = "gpt2"  # 124M parameters — small enough for a laptop

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)

print(f"Model: {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Vocabulary: {tokenizer.vocab_size:,} tokens")

In [ ]:
def generate(prompt, model, max_new_tokens=100, temperature=0.8):
    """Generate text from a prompt."""
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
# --- Baseline: GPT-2 BEFORE additional training ---
prompts = [
    "ROMEO: O, she doth teach the torches",
    "To be, or not to be,",
    "Friends, Romans, countrymen,",
]

print("=" * 70)
print("BEFORE continued pre-training")
print("=" * 70)
for prompt in prompts:
    text = generate(prompt, model)
    print(f"\nPrompt: \"{prompt}\"")
    print(text)
    print("-" * 70)

## 3 — Tokenize the Corpus

We need to convert our text file into tokenized chunks that the model can train on.

Each training example is a sequence of `block_size` tokens. The model learns to predict each token from the tokens before it (the causal LM objective).

In [ ]:
from datasets import load_dataset

BLOCK_SIZE = 128  # sequence length for training — longer = more context but more memory

# Load corpus as a HuggingFace dataset
dataset = load_dataset("text", data_files={"train": str(CORPUS_PATH)}, split="train")
print(f"Raw lines: {len(dataset):,}")

# Tokenize
def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=False)

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

# Group into blocks of BLOCK_SIZE
def group_texts(examples):
    # Concatenate all token ids
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated["input_ids"])
    # Drop the remainder
    total_length = (total_length // BLOCK_SIZE) * BLOCK_SIZE
    result = {
        k: [v[i : i + BLOCK_SIZE] for i in range(0, total_length, BLOCK_SIZE)]
        for k, v in concatenated.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

lm_dataset = tokenized.map(group_texts, batched=True)
print(f"Training examples: {len(lm_dataset):,} sequences of {BLOCK_SIZE} tokens")
print(f"Total training tokens: {len(lm_dataset) * BLOCK_SIZE:,}")

In [ ]:
# Peek at a training example
example = lm_dataset[0]
decoded = tokenizer.decode(example["input_ids"])
print(f"Example training sequence ({BLOCK_SIZE} tokens):\n")
print(decoded[:500])

## 4 — Train!

We use HuggingFace's `Trainer` to continue pre-training. This is the same next-token prediction objective GPT-2 was originally trained with — we're just giving it more text to learn from.

**On CPU** this takes ~5-15 minutes for 3 epochs. On a GPU it's much faster.

Adjust `num_train_epochs` and `per_device_train_batch_size` based on your hardware.

In [ ]:
# ── Training configuration ──────────────────────────────────
OUTPUT_DIR = "./gpt2-shakespeare"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,   # reduce to 2 if you run out of memory
    learning_rate=5e-5,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),  # mixed precision on GPU
    report_to="none",               # disable wandb/tensorboard
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # GPT-2 is causal LM, not masked LM
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset,
    data_collator=data_collator,
)

print(f"Training config:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Device: {training_args.device}")

In [ ]:
%%time

print("Training started...\n")
trainer.train()
print("\nTraining complete!")

In [ ]:
import matplotlib.pyplot as plt

# Plot training loss
log_history = trainer.state.log_history
steps = [entry["step"] for entry in log_history if "loss" in entry]
losses = [entry["loss"] for entry in log_history if "loss" in entry]

plt.figure(figsize=(10, 4))
plt.plot(steps, losses, linewidth=2)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training Loss — Continued Pre-training on Shakespeare")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Initial loss: {losses[0]:.3f}")
print(f"Final loss:   {losses[-1]:.3f}")

## 5 — Generate After Training

Now let's use the **same prompts** and see how the model's style has shifted.

In [ ]:
print("=" * 70)
print("AFTER continued pre-training on Shakespeare")
print("=" * 70)
for prompt in prompts:
    text = generate(prompt, model)
    print(f"\nPrompt: \"{prompt}\"")
    print(text)
    print("-" * 70)

In [ ]:
# Side-by-side comparison with the original model
original_model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)

test_prompt = "JULIET: What light through yonder"

print(f'Prompt: "{test_prompt}"\n')
print("--- Original GPT-2 ---")
print(generate(test_prompt, original_model))
print("\n--- After Shakespeare training ---")
print(generate(test_prompt, model))

## 6 — Perplexity: Measuring How Well the Model Learned

**Perplexity** measures how "surprised" the model is by the text. Lower = better.

A perplexity of 20 means the model is, on average, as uncertain as if it were choosing among 20 equally likely words.

In [ ]:
import math

def compute_perplexity(text, model, max_length=512):
    """Compute perplexity of a model on a text sample."""
    encodings = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length).to(device)
    with torch.no_grad():
        outputs = model(**encodings, labels=encodings["input_ids"])
    return math.exp(outputs.loss.item())

# Test on a Shakespeare passage not in the first part of training
test_text = text[-2000:]  # last 2000 characters of the corpus

ppl_original = compute_perplexity(test_text, original_model)
ppl_finetuned = compute_perplexity(test_text, model)

print(f"Perplexity on Shakespeare text:")
print(f"  Original GPT-2:  {ppl_original:.1f}")
print(f"  After training:  {ppl_finetuned:.1f}")
print(f"  Improvement:     {(1 - ppl_finetuned/ppl_original)*100:.1f}%")

## 7 — Save & Reload the Model

In [ ]:
SAVE_DIR = "./gpt2-shakespeare-final"

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Model saved to {SAVE_DIR}/")

# Reload to verify
reloaded = GPT2LMHeadModel.from_pretrained(SAVE_DIR).to(device)
print(generate("HAMLET: To be, or not to be,", reloaded))

## 8 — What We Just Did

| Step | What | Why |
|---|---|---|
| Load GPT-2 | Start from a model that already understands English | Transfer learning — don't start from scratch |
| Tokenize corpus | Convert Shakespeare text into token sequences | The model trains on tokens, not raw text |
| Train (causal LM) | Predict next token on Shakespeare text | The model learns Shakespeare's vocabulary, style, and patterns |
| Generate | Sample from the updated model | See how the style shifted toward Shakespeare |

This is **continued pre-training** (also called domain-adaptive pre-training). The same idea is used to:
- Train medical LLMs on PubMed articles
- Train code models on GitHub repositories
- Train legal models on court documents

**Next step:** fine-tuning for a specific task (instruction following, Q&A, etc.) → see `04_chat_models.ipynb`